# DriftGuard — Environment and Repository Setup

This notebook:

1. Verifies the Python and Conda environment.
2. Creates the required project directories.
3. Defines training, validation and unseen test repositories.
4. Clones complete Git histories.
5. Records repository versions for reproducibility.

In [1]:
import os
import sys
import json
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("DRIFTGUARD ENVIRONMENT CHECK")
print("=" * 70)

print("Python version :", sys.version)
print("Python path    :", sys.executable)
print("Platform       :", platform.platform())
print("Working folder:", Path.cwd())
print("Conda env      :", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))

expected_environment = "driftguard"
current_environment = os.environ.get("CONDA_DEFAULT_ENV", "").lower()

if current_environment == expected_environment:
    print("\nEnvironment check: PASSED")
else:
    print(
        f"\nWarning: Expected Conda environment '{expected_environment}', "
        f"but detected '{current_environment or 'unknown'}'."
    )

DRIFTGUARD ENVIRONMENT CHECK
Python version : 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Python path    : C:\Users\Lenovo\anaconda3\envs\driftguard\python.exe
Platform       : Windows-10-10.0.26200-SP0
Working folder: C:\Users\Lenovo\Desktop\DriftGuard\notebooks
Conda env      : driftguard

Environment check: PASSED


In [2]:
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "yaml": "pyyaml",
    "git": "gitpython",
    "pydriller": "pydriller",
    "tqdm": "tqdm",
    "joblib": "joblib",
    "matplotlib": "matplotlib",
}

missing_packages = []

for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(package_name)

if missing_packages:
    print("Missing packages:")
    for package in missing_packages:
        print(" -", package)

    print("\nRun the next installation cell.")
else:
    print("All required packages are installed.")

All required packages are installed.


In [2]:
%pip install pandas numpy scikit-learn pyyaml gitpython pydriller tqdm joblib matplotlib requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

print("Project root:", PROJECT_ROOT)

if PROJECT_ROOT.name.lower() != "driftguard":
    print(
        "\nWarning: The detected folder is not named DriftGuard.\n"
        "The notebook will still work, but confirm that it is saved inside "
        "your DriftGuard project."
    )

Project root: C:\Users\Lenovo\Desktop\DriftGuard


In [4]:
PROJECT_DIRECTORIES = [
    "notebooks",
    "repositories",
    "data/raw",
    "data/interim",
    "data/processed",
    "data/labels",
    "data/evaluation",
    "models",
    "outputs",
    "outputs/figures",
    "outputs/metrics",
    "reports",
    "src",
    "src/ingestion",
    "src/parsers",
    "src/features",
    "src/models",
    "src/rules",
    "src/evaluation",
    "tests",
    "configs",
    "logs",
]

for relative_directory in PROJECT_DIRECTORIES:
    directory = PROJECT_ROOT / relative_directory
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully.\n")

for relative_directory in PROJECT_DIRECTORIES:
    directory = PROJECT_ROOT / relative_directory
    print(f"[OK] {directory.relative_to(PROJECT_ROOT)}")

Project directories created successfully.

[OK] notebooks
[OK] repositories
[OK] data\raw
[OK] data\interim
[OK] data\processed
[OK] data\labels
[OK] data\evaluation
[OK] models
[OK] outputs
[OK] outputs\figures
[OK] outputs\metrics
[OK] reports
[OK] src
[OK] src\ingestion
[OK] src\parsers
[OK] src\features
[OK] src\models
[OK] src\rules
[OK] src\evaluation
[OK] tests
[OK] configs
[OK] logs


In [5]:
RANDOM_STATE = 42

CONFIG_EXTENSIONS = {
    ".yaml",
    ".yml",
    ".json",
    ".tf",
    ".tfvars",
    ".hcl",
    ".conf",
    ".cfg",
    ".ini",
    ".toml",
    ".properties",
}

CONFIG_FILENAMES = {
    "dockerfile",
    "docker-compose.yml",
    "docker-compose.yaml",
    "ansible.cfg",
    "nginx.conf",
    "httpd.conf",
}

MAX_FILE_SIZE_BYTES = 1_000_000

PROJECT_SETTINGS = {
    "project_name": "DriftGuard",
    "random_state": RANDOM_STATE,
    "config_extensions": sorted(CONFIG_EXTENSIONS),
    "config_filenames": sorted(CONFIG_FILENAMES),
    "max_file_size_bytes": MAX_FILE_SIZE_BYTES,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

settings_path = PROJECT_ROOT / "configs" / "project_settings.json"

with settings_path.open("w", encoding="utf-8") as file:
    json.dump(PROJECT_SETTINGS, file, indent=2)

print("Settings saved to:")
print(settings_path)
print("\nSettings:")
print(json.dumps(PROJECT_SETTINGS, indent=2))

Settings saved to:
C:\Users\Lenovo\Desktop\DriftGuard\configs\project_settings.json

Settings:
{
  "project_name": "DriftGuard",
  "random_state": 42,
  "config_extensions": [
    ".cfg",
    ".conf",
    ".hcl",
    ".ini",
    ".json",
    ".properties",
    ".tf",
    ".tfvars",
    ".toml",
    ".yaml",
    ".yml"
  ],
  "config_filenames": [
    "ansible.cfg",
    "docker-compose.yaml",
    "docker-compose.yml",
    "dockerfile",
    "httpd.conf",
    "nginx.conf"
  ],
  "max_file_size_bytes": 1000000,
  "created_at_utc": "2026-07-19T18:39:09.616100+00:00"
}


In [6]:
REPOSITORIES = [
    {
        "name": "microservices_demo",
        "url": "https://github.com/GoogleCloudPlatform/microservices-demo.git",
        "split": "train",
        "primary_config_types": [
            "kubernetes",
            "helm",
            "kustomize",
            "terraform",
        ],
    },
    {
        "name": "kube_prometheus",
        "url": "https://github.com/prometheus-operator/kube-prometheus.git",
        "split": "train",
        "primary_config_types": [
            "kubernetes",
            "jsonnet",
            "yaml",
        ],
    },
    {
        "name": "terraform_aws_vpc",
        "url": "https://github.com/terraform-aws-modules/terraform-aws-vpc.git",
        "split": "train",
        "primary_config_types": [
            "terraform",
        ],
    },
    {
        "name": "kubernetes_examples",
        "url": "https://github.com/kubernetes/examples.git",
        "split": "validation",
        "primary_config_types": [
            "kubernetes",
            "yaml",
        ],
    },
    {
        "name": "ansible_examples",
        "url": "https://github.com/ansible/ansible-examples.git",
        "split": "test",
        "primary_config_types": [
            "ansible",
            "nginx",
            "yaml",
        ],
    },
    {
        "name": "terraform_eks_blueprints",
        "url": "https://github.com/aws-ia/terraform-aws-eks-blueprints.git",
        "split": "test",
        "primary_config_types": [
            "terraform",
            "kubernetes",
        ],
    },
]

split_counts = {}

for repository in REPOSITORIES:
    split = repository["split"]
    split_counts[split] = split_counts.get(split, 0) + 1

print("Repository split design:")
print(json.dumps(split_counts, indent=2))

assert split_counts.get("train", 0) >= 3
assert split_counts.get("validation", 0) >= 1
assert split_counts.get("test", 0) >= 2

print("\nRepository-disjoint split check: PASSED")

Repository split design:
{
  "train": 3,
  "validation": 1,
  "test": 2
}

Repository-disjoint split check: PASSED


In [7]:
repository_config_path = PROJECT_ROOT / "configs" / "repositories.json"

repository_configuration = {
    "description": (
        "Public Git repositories used for DriftGuard model development. "
        "Test repositories remain completely unseen during training."
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repositories": REPOSITORIES,
}

with repository_config_path.open("w", encoding="utf-8") as file:
    json.dump(repository_configuration, file, indent=2)

print("Repository configuration saved to:")
print(repository_config_path)

Repository configuration saved to:
C:\Users\Lenovo\Desktop\DriftGuard\configs\repositories.json


In [8]:
def run_command(command, cwd=None, check=True):
    """Run a command and return its completed process."""

    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(command)}\n"
            f"Standard output:\n{result.stdout}\n"
            f"Error output:\n{result.stderr}"
        )

    return result


git_version_result = run_command(["git", "--version"])

print("Git installation:", git_version_result.stdout.strip())
print("Git check: PASSED")

Git installation: git version 2.51.2.windows.1
Git check: PASSED


In [9]:
def check_remote_repository(repository_url):
    result = run_command(
        ["git", "ls-remote", "--heads", repository_url],
        check=False,
    )

    return {
        "accessible": result.returncode == 0,
        "error": result.stderr.strip(),
    }


accessibility_results = []

for repository in REPOSITORIES:
    result = check_remote_repository(repository["url"])

    accessibility_results.append(
        {
            "name": repository["name"],
            "split": repository["split"],
            "accessible": result["accessible"],
            "error": result["error"],
        }
    )

    status = "ACCESSIBLE" if result["accessible"] else "FAILED"
    print(f"{repository['name']:<30} {status}")

failed_repositories = [
    result
    for result in accessibility_results
    if not result["accessible"]
]

if failed_repositories:
    print("\nSome repositories could not be reached:")
    for repository in failed_repositories:
        print(repository["name"], ":", repository["error"])
else:
    print("\nAll repository URLs are accessible.")

microservices_demo             ACCESSIBLE
kube_prometheus                ACCESSIBLE
terraform_aws_vpc              ACCESSIBLE
kubernetes_examples            ACCESSIBLE
ansible_examples               ACCESSIBLE
terraform_eks_blueprints       ACCESSIBLE

All repository URLs are accessible.


In [10]:
REPOSITORIES_DIR = PROJECT_ROOT / "repositories"

def clone_or_validate_repository(repository):
    name = repository["name"]
    url = repository["url"]
    destination = REPOSITORIES_DIR / name

    print("\n" + "=" * 70)
    print(f"Repository: {name}")
    print(f"Split     : {repository['split']}")
    print(f"Location  : {destination}")
    print("=" * 70)

    if (destination / ".git").exists():
        print("Repository already exists. Skipping clone.")
        status = "existing"

    elif destination.exists():
        print(
            "Destination exists but is not a valid Git repository. "
            "It will not be overwritten."
        )
        return {
            "name": name,
            "split": repository["split"],
            "status": "invalid_existing_directory",
            "path": str(destination),
            "error": "Directory exists without a .git folder.",
        }

    else:
        command = [
            "git",
            "clone",
            "--single-branch",
            "--no-tags",
            url,
            str(destination),
        ]

        result = run_command(command, check=False)

        if result.returncode != 0:
            print("Clone failed.")
            print(result.stderr)

            return {
                "name": name,
                "split": repository["split"],
                "status": "clone_failed",
                "path": str(destination),
                "error": result.stderr.strip(),
            }

        print("Clone completed.")
        status = "cloned"

    return {
        "name": name,
        "split": repository["split"],
        "status": status,
        "path": str(destination),
        "error": "",
    }


clone_results = []

for repository in REPOSITORIES:
    clone_result = clone_or_validate_repository(repository)
    clone_results.append(clone_result)

print("\nClone stage completed.")


Repository: microservices_demo
Split     : train
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\microservices_demo
Clone completed.

Repository: kube_prometheus
Split     : train
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\kube_prometheus
Clone completed.

Repository: terraform_aws_vpc
Split     : train
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\terraform_aws_vpc
Clone completed.

Repository: kubernetes_examples
Split     : validation
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\kubernetes_examples
Clone completed.

Repository: ansible_examples
Split     : test
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\ansible_examples
Clone completed.

Repository: terraform_eks_blueprints
Split     : test
Location  : C:\Users\Lenovo\Desktop\DriftGuard\repositories\terraform_eks_blueprints
Clone completed.

Clone stage completed.


In [11]:
import pandas as pd

def get_git_value(repository_path, arguments):
    result = run_command(
        ["git", "-C", str(repository_path), *arguments],
        check=False,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


manifest_records = []

for repository in REPOSITORIES:
    repository_path = REPOSITORIES_DIR / repository["name"]

    if not (repository_path / ".git").exists():
        manifest_records.append(
            {
                "repository": repository["name"],
                "split": repository["split"],
                "url": repository["url"],
                "available": False,
                "branch": None,
                "head_commit": None,
                "commit_count": None,
                "oldest_commit_date": None,
                "latest_commit_date": None,
                "local_path": str(repository_path),
            }
        )
        continue

    branch = get_git_value(
        repository_path,
        ["branch", "--show-current"],
    )

    head_commit = get_git_value(
        repository_path,
        ["rev-parse", "HEAD"],
    )

    commit_count = get_git_value(
        repository_path,
        ["rev-list", "--count", "HEAD"],
    )

    oldest_commit_date = get_git_value(
        repository_path,
        [
            "log",
            "--reverse",
            "--format=%cI",
            "-1",
        ],
    )

    latest_commit_date = get_git_value(
        repository_path,
        [
            "log",
            "-1",
            "--format=%cI",
        ],
    )

    manifest_records.append(
        {
            "repository": repository["name"],
            "split": repository["split"],
            "url": repository["url"],
            "available": True,
            "branch": branch,
            "head_commit": head_commit,
            "commit_count": (
                int(commit_count)
                if commit_count and commit_count.isdigit()
                else None
            ),
            "oldest_commit_date": oldest_commit_date,
            "latest_commit_date": latest_commit_date,
            "local_path": str(repository_path),
        }
    )

repository_manifest = pd.DataFrame(manifest_records)

repository_manifest

,repository,split,url,available,branch,head_commit,commit_count,oldest_commit_date,latest_commit_date,local_path
0,microservices_demo,train,https://github.com/GoogleCloudPlatform/microse...,True,main,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2644,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
1,kube_prometheus,train,https://github.com/prometheus-operator/kube-pr...,True,main,f912700123916f3ecc80bbb6dd61868cabe9a02f,3354,2026-07-18T10:34:10+05:30,2026-07-18T10:34:10+05:30,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
2,terraform_aws_vpc,train,https://github.com/terraform-aws-modules/terra...,True,master,3ffbd46fb1c7733e1b34d8666893280454e27436,519,2026-04-02T20:22:06Z,2026-04-02T20:22:06Z,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
3,kubernetes_examples,validation,https://github.com/kubernetes/examples.git,True,master,d6b8cd27eacb51e651a1aa6f7c190a28713eff6e,2103,2026-03-03T23:25:19+05:30,2026-03-03T23:25:19+05:30,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
4,ansible_examples,test,https://github.com/ansible/ansible-examples.git,True,master,b50586543c6c4be907fdc88f9f78a2b35d2a895f,439,2020-08-01T11:36:58+02:00,2020-08-01T11:36:58+02:00,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
5,terraform_eks_blueprints,test,https://github.com/aws-ia/terraform-aws-eks-bl...,True,main,eec18146cc2d883e093c548de0a92a5d510205d1,1555,2026-07-07T18:04:37+02:00,2026-07-07T18:04:37+02:00,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...


In [12]:
manifest_csv_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "repository_manifest.csv"
)

manifest_json_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "repository_manifest.json"
)

repository_manifest.to_csv(
    manifest_csv_path,
    index=False,
)

with manifest_json_path.open("w", encoding="utf-8") as file:
    json.dump(
        repository_manifest.to_dict(orient="records"),
        file,
        indent=2,
        default=str,
    )

print("Repository manifest saved successfully.")
print("CSV :", manifest_csv_path)
print("JSON:", manifest_json_path)

Repository manifest saved successfully.
CSV : C:\Users\Lenovo\Desktop\DriftGuard\data\raw\repository_manifest.csv
JSON: C:\Users\Lenovo\Desktop\DriftGuard\data\raw\repository_manifest.json


In [13]:
available_manifest = repository_manifest[
    repository_manifest["available"] == True
].copy()

available_split_counts = (
    available_manifest["split"]
    .value_counts()
    .to_dict()
)

print("Available repositories by split:")
print(available_split_counts)

checks = {
    "At least 3 training repositories":
        available_split_counts.get("train", 0) >= 3,

    "At least 1 validation repository":
        available_split_counts.get("validation", 0) >= 1,

    "At least 2 unseen test repositories":
        available_split_counts.get("test", 0) >= 2,

    "All repositories have HEAD commits":
        available_manifest["head_commit"].notna().all(),

    "All repositories have commit histories":
        (available_manifest["commit_count"].fillna(0) > 1).all(),
}

print("\nValidation checks:")

for check_name, passed in checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status:<8} | {check_name}")

all_checks_passed = all(checks.values())

print("\n" + "=" * 70)

if all_checks_passed:
    print("NOTEBOOK 01 COMPLETED SUCCESSFULLY")
else:
    print("NOTEBOOK 01 COMPLETED WITH FAILED CHECKS")

print("=" * 70)

Available repositories by split:
{'train': 3, 'test': 2, 'validation': 1}

Validation checks:
PASSED   | At least 3 training repositories
PASSED   | At least 1 validation repository
PASSED   | At least 2 unseen test repositories
PASSED   | All repositories have HEAD commits
PASSED   | All repositories have commit histories

NOTEBOOK 01 COMPLETED SUCCESSFULLY


In [14]:
print("Project root:")
print(PROJECT_ROOT)

print("\nRepository storage:")
print(REPOSITORIES_DIR)

print("\nRepository manifest:")
display(
    repository_manifest[
        [
            "repository",
            "split",
            "available",
            "branch",
            "commit_count",
            "head_commit",
        ]
    ]
)

total_commits = (
    repository_manifest["commit_count"]
    .fillna(0)
    .sum()
)

print(f"\nTotal commits available across repositories: {int(total_commits):,}")
print("\nNext notebook: 02_git_history_extraction.ipynb")

Project root:
C:\Users\Lenovo\Desktop\DriftGuard

Repository storage:
C:\Users\Lenovo\Desktop\DriftGuard\repositories

Repository manifest:


,repository,split,available,branch,commit_count,head_commit
0,microservices_demo,train,True,main,2644,9a4616e77f0f9cbcbecaf27d711c38890dda1404
1,kube_prometheus,train,True,main,3354,f912700123916f3ecc80bbb6dd61868cabe9a02f
2,terraform_aws_vpc,train,True,master,519,3ffbd46fb1c7733e1b34d8666893280454e27436
3,kubernetes_examples,validation,True,master,2103,d6b8cd27eacb51e651a1aa6f7c190a28713eff6e
4,ansible_examples,test,True,master,439,b50586543c6c4be907fdc88f9f78a2b35d2a895f
5,terraform_eks_blueprints,test,True,main,1555,eec18146cc2d883e093c548de0a92a5d510205d1



Total commits available across repositories: 10,614

Next notebook: 02_git_history_extraction.ipynb
